# Phase 3 Exploration: Skema Tagging (`BIO` vs `BIOES`) & Tokenizer (`WordPiece` vs `BPE` vs `SentencePiece`)

Eksperimen terarah untuk membandingkan 6 kombinasi (`BIO`/`BIOES` x `bert-base-uncased`/`roberta-base`/`xlm-roberta-base`) pada `tsd_train` dan `tsd_trial` (`tsd_test` tidak disentuh):
1. **Bagian A (Intrinsik):** *Round-trip Character F1*, distribusi tag, fragmentasi subword, dan rasio terpotong `MAX_LEN = 256`.
2. **Bagian B (Non-Sekuensial Cepat):** `SGDClassifier` berbasis fitur subword bersih & konteks lokal ($t-1, t, t+1$).
3. **Bagian C (Sekuensial Ringan):** `Embedding + BiLSTM + torchcrf.CRF` untuk menguji dampak matriks transisi `BIO` vs `BIOES` secara adil.

In [1]:
from collections import Counter
from pathlib import Path
import sys
import time

import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchcrf import CRF
from transformers import AutoTokenizer

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src" / "task_3").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.task_3.config import MAX_LEN, SEEDS, set_seed
from src.task_3.dataset import _contiguous_ranges, load_semeval_split
from src.task_3.metrics import evaluate_bio_tags, evaluate_char_spans


def log(tag, message, *, err=None):
    if err is None:
        err = tag in ("ERROR", "FATAL", "WARN")
    print(f"[{tag}] {message}", file=sys.stderr if err else sys.stdout, flush=True)


def print_table(title, rows, line_width=49):
    print(title)
    widths = [max(len(str(c)) for c in col) for col in zip(*rows)]
    total = sum(widths) + 3 * (len(widths) - 1)
    print("-" * max(line_width, total))
    for row in rows:
        print(" | ".join(str(c).ljust(w) for c, w in zip(row, widths)).rstrip())


set_seed(SEEDS[0])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SCHEMES = {
    "BIO": {"O": 0, "B-TOXIC": 1, "I-TOXIC": 2},
    "BIOES": {"O": 0, "B-TOXIC": 1, "I-TOXIC": 2, "E-TOXIC": 3, "S-TOXIC": 4},
}
TOKENIZER_SPECS = {
    "BERT-WordPiece": "bert-base-uncased",
    "RoBERTa-BPE": "roberta-base",
    "XLM-R-SPM": "xlm-roberta-base",
}

train_rows = load_semeval_split("train")
trial_rows = load_semeval_split("trial")
log("START", f"Loaded train={len(train_rows)}, trial={len(trial_rows)}, device={DEVICE}")

[START] Loaded train=7939, trial=690, device=cuda


In [2]:
def trim_offset(text, start, end):
    while start < end and text[start].isspace():
        start += 1
    while end > start and text[end - 1].isspace():
        end -= 1
    return start, end


def encode_split(rows, tokenizer, scheme, max_length=MAX_LEN):
    label2id = SCHEMES[scheme]
    texts = [r["text"] for r in rows]
    full_enc = tokenizer(texts, truncation=False, padding=False)
    truncated_count = sum(len(ids) > max_length for ids in full_enc["input_ids"])

    enc = tokenizer(
        texts,
        return_offsets_mapping=True,
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    samples = []
    for idx, row in enumerate(rows):
        text = row["text"]
        offsets = enc["offset_mapping"][idx]
        seq_ids = enc.sequence_ids(idx)

        ranges = _contiguous_ranges(sorted(set(row["spans"])))
        char_to_range = {}
        for r_idx, (s, e) in enumerate(ranges):
            for pos in range(s, e + 1):
                char_to_range[pos] = r_idx

        token_ranges = [None] * len(offsets)
        is_ws_only = [False] * len(offsets)
        trimmed_offsets = []

        for t_idx, (orig_s, orig_e) in enumerate(offsets):
            if seq_ids[t_idx] is None or orig_s == orig_e:
                trimmed_offsets.append((0, 0))
                continue
            ts, te = trim_offset(text, orig_s, orig_e)
            trimmed_offsets.append((ts, te))
            if ts == te:
                is_ws_only[t_idx] = True
                continue
            matched = [char_to_range[p] for p in range(ts, te) if p in char_to_range]
            if len(matched) * 2 >= (te - ts):
                token_ranges[t_idx] = Counter(matched).most_common(1)[0][0]

        non_ws_indices = [i for i, ws in enumerate(is_ws_only) if not ws]
        labels = [label2id["O"]] * len(offsets)

        for pos, t_idx in enumerate(non_ws_indices):
            cur_r = token_ranges[t_idx]
            if cur_r is None:
                continue
            prev_r = token_ranges[non_ws_indices[pos - 1]] if pos > 0 else None
            next_r = token_ranges[non_ws_indices[pos + 1]] if pos + 1 < len(non_ws_indices) else None
            same_prev = prev_r == cur_r
            same_next = next_r == cur_r

            if scheme == "BIO":
                tag = "I-TOXIC" if same_prev else "B-TOXIC"
            else:
                if not same_prev and not same_next:
                    tag = "S-TOXIC"
                elif not same_prev and same_next:
                    tag = "B-TOXIC"
                elif same_prev and same_next:
                    tag = "I-TOXIC"
                else:
                    tag = "E-TOXIC"
            labels[t_idx] = label2id[tag]

        samples.append({
            "input_ids": enc["input_ids"][idx],
            "attention_mask": enc["attention_mask"][idx],
            "labels": labels,
            "offset_mapping": offsets,
            "trimmed_offsets": trimmed_offsets,
            "n_ranges": len(ranges),
        })

    return samples, truncated_count


def decode_spans(tag_ids, offset_mapping, text, id2label):
    spans = []
    prev_toxic_end = None

    for t_id, (orig_s, orig_e) in zip(tag_ids, offset_mapping):
        if orig_s == orig_e:
            prev_toxic_end = None
            continue
        ts, te = trim_offset(text, orig_s, orig_e)
        if ts == te:
            continue

        tag = id2label[int(t_id)]
        if tag == "O":
            prev_toxic_end = None
            continue

        if (
            tag in ("I-TOXIC", "E-TOXIC")
            and prev_toxic_end is not None
            and ts > prev_toxic_end
            and text[prev_toxic_end:ts].isspace()
        ):
            spans.extend(range(prev_toxic_end, ts))

        spans.extend(range(ts, te))
        prev_toxic_end = te

    return sorted(set(spans))


tokenizers = {
    name: AutoTokenizer.from_pretrained(hf_id, use_fast=True)
    for name, hf_id in TOKENIZER_SPECS.items()
}

sanity_row = [{"text": "Because he is a violent and aggressive idiot.", "spans": list(range(16, 44))}]
for tok_name, tok in tokenizers.items():
    assert tok.is_fast, f"{tok_name} is not fast"
    for scheme_name, label2id in SCHEMES.items():
        id2label = {v: k for k, v in label2id.items()}
        enc_s, _ = encode_split(sanity_row, tok, scheme_name)
        rec = decode_spans(enc_s[0]["labels"], enc_s[0]["offset_mapping"], sanity_row[0]["text"], id2label)
        assert rec == sanity_row[0]["spans"], f"Sanity mismatch on {tok_name} {scheme_name}: {rec}"

log("CHECKPOINT 1", "3 fast tokenizers termuat dan sanity check BIO/BIOES lulus")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

c:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

[CHECKPOINT 1] 3 fast tokenizers termuat dan sanity check BIO/BIOES lulus


## Bagian A — Uji Intrinsik & *Round-Trip Sanity Check* (Tanpa Model)

In [3]:
encoded_cache = {}
part_a_rows = [["Tokenizer", "Scheme", "Train RT-F1", "Trial RT-F1", "Trial Empty", "Trial NonEmpty", "Subw/Span", "Trunc(Tr)", "ToxTok%"]]
tag_dist_rows = [["Tokenizer", "Scheme", "O", "B-TOXIC", "I-TOXIC", "E-TOXIC", "S-TOXIC"]]

for tok_name, tok in tokenizers.items():
    for scheme_name, label2id in SCHEMES.items():
        id2label = {v: k for k, v in label2id.items()}
        tr_samples, tr_trunc = encode_split(train_rows, tok, scheme_name)
        va_samples, va_trunc = encode_split(trial_rows, tok, scheme_name)
        encoded_cache[(tok_name, scheme_name)] = {"train": tr_samples, "trial": va_samples}

        tr_rec = [decode_spans(s["labels"], s["offset_mapping"], r["text"], id2label) for r, s in zip(train_rows, tr_samples)]
        va_rec = [decode_spans(s["labels"], s["offset_mapping"], r["text"], id2label) for r, s in zip(trial_rows, va_samples)]
        tr_stat = evaluate_char_spans(tr_rec, [r["spans"] for r in train_rows])
        va_stat = evaluate_char_spans(va_rec, [r["spans"] for r in trial_rows])

        counts = Counter(id2label[lbl] for s in tr_samples for lbl in s["labels"])
        total_toks = sum(counts.values())
        tox_toks = total_toks - counts["O"]
        total_ranges = sum(s["n_ranges"] for s in tr_samples)

        part_a_rows.append([
            tok_name,
            scheme_name,
            f"{tr_stat['f1']:.5f}",
            f"{va_stat['f1']:.5f}",
            f"{va_stat['empty_f1']:.5f}",
            f"{va_stat['non_empty_f1']:.5f}",
            f"{tox_toks / max(total_ranges, 1):.2f}",
            str(tr_trunc),
            f"{100.0 * tox_toks / total_toks:.2f}%",
        ])
        tag_dist_rows.append([
            tok_name,
            scheme_name,
            str(counts["O"]),
            str(counts["B-TOXIC"]),
            str(counts["I-TOXIC"]),
            str(counts.get("E-TOXIC", "-")),
            str(counts.get("S-TOXIC", "-")),
        ])

log("CHECKPOINT 2", "Bagian A (Round-Trip 6 kombinasi) selesai")
print_table("Bagian A: Round-Trip F1 & Statistik Subword", part_a_rows)
print()
print_table("Distribusi Tag BIO vs BIOES pada tsd_train", tag_dist_rows)

[CHECKPOINT 2] Bagian A (Round-Trip 6 kombinasi) selesai
Bagian A: Round-Trip F1 & Statistik Subword
--------------------------------------------------------------------------------------------------------------------
Tokenizer      | Scheme | Train RT-F1 | Trial RT-F1 | Trial Empty | Trial NonEmpty | Subw/Span | Trunc(Tr) | ToxTok%
BERT-WordPiece | BIO    | 0.99875     | 0.99789     | 1.00000     | 0.99775        | 3.01      | 10        | 7.90%
BERT-WordPiece | BIOES  | 0.99875     | 0.99789     | 1.00000     | 0.99775        | 3.01      | 10        | 7.90%
RoBERTa-BPE    | BIO    | 0.99861     | 0.99764     | 1.00000     | 0.99748        | 2.95      | 14        | 7.78%
RoBERTa-BPE    | BIOES  | 0.99861     | 0.99764     | 1.00000     | 0.99748        | 2.95      | 14        | 7.78%
XLM-R-SPM      | BIO    | 0.99853     | 0.99808     | 1.00000     | 0.99796        | 3.62      | 33        | 8.77%
XLM-R-SPM      | BIOES  | 0.99853     | 0.99808     | 1.00000     | 0.99796        | 3.62 

## Bagian B — Probe Cepat Non-Sekuensial (`SGDClassifier`)

Mengukur kualitas representasi subword masing-masing tokenizer menggunakan fitur teks token bersih (`text[ts:te]`), konteks lokal ($t-1, t+1$), dan *char n-gram TF-IDF*. Karena `SGDClassifier` memprediksi setiap token secara independen, bagian ini terutama membaca pengaruh **tokenizer**, bukan matriks transisi `BIO` vs `BIOES`.

In [4]:
def extract_token_features(rows, samples):
    feat_dicts, surfs, labels, tok_positions = [], [], [], []
    for row_idx, (row, sample) in enumerate(zip(rows, samples)):
        text = row["text"]
        t_offsets = sample["trimmed_offsets"]
        valid_pos = [i for i, (ts, te) in enumerate(t_offsets) if ts < te]
        tokens = [text[t_offsets[i][0]:t_offsets[i][1]] for i in valid_pos]
        lows = [t.lower() for t in tokens]

        for k, t_idx in enumerate(valid_pos):
            ts, te = t_offsets[t_idx]
            raw_tok = tokens[k]
            surf = lows[k]
            feat_dicts.append({
                "w0": surf,
                "w-1": lows[k - 1] if k > 0 else "<BOS>",
                "w+1": lows[k + 1] if k + 1 < len(lows) else "<EOS>",
                "is_cont": int(ts > 0 and not text[ts - 1].isspace()),
                "is_upper": int(raw_tok.isupper()),
                "is_title": int(raw_tok.istitle()),
                "has_digit": int(any(c.isdigit() for c in raw_tok)),
                "len": min(len(raw_tok), 20),
            })
            surfs.append(surf)
            labels.append(sample["labels"][t_idx])
            tok_positions.append((row_idx, t_idx))
    return feat_dicts, surfs, np.array(labels, dtype=np.int64), tok_positions


part_b_rows = [["Tokenizer", "Scheme", "Char F1", "Empty F1", "NonEmpty F1", "Seqeval F1", "Time(s)"]]

for tok_name in tokenizers:
    for scheme_name, label2id in SCHEMES.items():
        t0 = time.perf_counter()
        id2label = {v: k for k, v in label2id.items()}
        tr_samples = encoded_cache[(tok_name, scheme_name)]["train"]
        va_samples = encoded_cache[(tok_name, scheme_name)]["trial"]

        tr_dicts, tr_surfs, y_tr, _ = extract_token_features(train_rows, tr_samples)
        va_dicts, va_surfs, y_va, va_pos = extract_token_features(trial_rows, va_samples)

        dv = DictVectorizer()
        tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=12000)
        X_tr = hstack([dv.fit_transform(tr_dicts), tfidf.fit_transform(tr_surfs)])
        X_va = hstack([dv.transform(va_dicts), tfidf.transform(va_surfs)])

        clf = SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            class_weight="balanced",
            max_iter=20,
            tol=1e-3,
            random_state=SEEDS[0],
        )
        clf.fit(X_tr, y_tr)
        pred_flat = clf.predict(X_va)

        pred_seqs = [[0] * len(s["labels"]) for s in va_samples]
        for (r_idx, t_idx), p_lbl in zip(va_pos, pred_flat):
            pred_seqs[r_idx][t_idx] = int(p_lbl)

        pred_spans = [
            decode_spans(p_seq, s["offset_mapping"], r["text"], id2label)
            for p_seq, s, r in zip(pred_seqs, va_samples, trial_rows)
        ]
        char_stat = evaluate_char_spans(pred_spans, [r["spans"] for r in trial_rows])
        pred_tags = [[id2label[t] for t in p_seq] for p_seq in pred_seqs]
        gold_tags = [[id2label[t] for t in s["labels"]] for s in va_samples]
        seq_stat = evaluate_bio_tags(pred_tags, gold_tags)

        part_b_rows.append([
            tok_name,
            scheme_name,
            f"{char_stat['f1']:.5f}",
            f"{char_stat['empty_f1']:.5f}",
            f"{char_stat['non_empty_f1']:.5f}",
            f"{seq_stat['f1']:.5f}",
            f"{time.perf_counter() - t0:.1f}",
        ])

log("CHECKPOINT 3", "Bagian B (SGDClassifier) selesai")
print_table("Bagian B: Probe Non-Sekuensial (SGDClassifier di tsd_trial)", part_b_rows)

c:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
c:\Users\ASUS\Desktop\nlp\tubes\IF5153-Chat-Moderation\venv\Lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:741: ConvergenceWarning: Maximum number of iteration reached before

[CHECKPOINT 3] Bagian B (SGDClassifier) selesai
Bagian B: Probe Non-Sekuensial (SGDClassifier di tsd_trial)
---------------------------------------------------------------------------------
Tokenizer      | Scheme | Char F1 | Empty F1 | NonEmpty F1 | Seqeval F1 | Time(s)
BERT-WordPiece | BIO    | 0.62743 | 0.02326  | 0.66758     | 0.48841    | 20.9
BERT-WordPiece | BIOES  | 0.51191 | 0.02326  | 0.54438     | 0.31982    | 27.9
RoBERTa-BPE    | BIO    | 0.56527 | 0.00000  | 0.60284     | 0.36851    | 19.9
RoBERTa-BPE    | BIOES  | 0.53743 | 0.00000  | 0.57315     | 0.33876    | 26.7
XLM-R-SPM      | BIO    | 0.59958 | 0.00000  | 0.63943     | 0.42258    | 22.8
XLM-R-SPM      | BIOES  | 0.55711 | 0.00000  | 0.59413     | 0.33263    | 28.0


## Bagian C — Probe Sekuensial Ringan (`Embedding + BiLSTM + CRF`)

Model sekuensial ringan ber-CRF untuk menguji kontribusi matriks transisi `BIO` vs `BIOES` bersama ketiga tokenizer pada `tsd_trial`.

In [5]:
class EncodedListDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def make_collate_fn(pad_token_id):
    pad_id = pad_token_id if pad_token_id is not None else 0

    def collate(batch):
        max_len = max(len(x["input_ids"]) for x in batch)
        ids, mask, labels, offsets = [], [], [], []
        for x in batch:
            pad_len = max_len - len(x["input_ids"])
            ids.append(x["input_ids"] + [pad_id] * pad_len)
            mask.append([bool(m) for m in x["attention_mask"]] + [False] * pad_len)
            labels.append(x["labels"] + [0] * pad_len)
            offsets.append(x["offset_mapping"])
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.bool),
            "labels": torch.tensor(labels, dtype=torch.long),
            "offset_mapping": offsets,
        }

    return collate


class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, num_tags, pad_idx, emb_dim=96, hidden_dim=96, dropout=0.25):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(hidden_dim * 2, num_tags)
        self.crf = CRF(num_tags=num_tags, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        emb = self.dropout(self.embedding(input_ids))
        seq_out, _ = self.lstm(emb)
        emissions = self.classifier(self.dropout(seq_out))
        if labels is not None:
            return -self.crf(emissions, labels, mask=attention_mask, reduction="mean")
        return self.crf.decode(emissions, mask=attention_mask)


def run_bilstm_crf(tok_name, tok, scheme_name, epochs=6, batch_size=64, lr=3e-3):
    set_seed(SEEDS[0])
    t0 = time.perf_counter()
    label2id = SCHEMES[scheme_name]
    id2label = {v: k for k, v in label2id.items()}
    tr_samples = encoded_cache[(tok_name, scheme_name)]["train"]
    va_samples = encoded_cache[(tok_name, scheme_name)]["trial"]

    collate_fn = make_collate_fn(tok.pad_token_id)
    tr_loader = DataLoader(EncodedListDataset(tr_samples), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    va_loader = DataLoader(EncodedListDataset(va_samples), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    pad_idx = tok.pad_token_id if tok.pad_token_id is not None else 0
    model = BiLSTMCRF(len(tok), len(label2id), pad_idx=pad_idx).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    best = {"char_f1": -1.0}
    trial_golds = [r["spans"] for r in trial_rows]

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in tr_loader:
            optimizer.zero_grad()
            loss = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE),
                batch["labels"].to(DEVICE),
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        pred_seqs = []
        with torch.no_grad():
            for batch in va_loader:
                pred_seqs.extend(model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE)))

        pred_spans = [
            decode_spans(p_seq, s["offset_mapping"], r["text"], id2label)
            for p_seq, s, r in zip(pred_seqs, va_samples, trial_rows)
        ]
        char_stat = evaluate_char_spans(pred_spans, trial_golds)
        if char_stat["f1"] > best["char_f1"]:
            pred_tags = [[id2label[t] for t in p_seq] for p_seq in pred_seqs]
            gold_tags = [[id2label[t] for t in s["labels"][:len(p_seq)]] for p_seq, s in zip(pred_seqs, va_samples)]
            seq_stat = evaluate_bio_tags(pred_tags, gold_tags)
            best = {
                "epoch": epoch,
                "char_f1": char_stat["f1"],
                "empty_f1": char_stat["empty_f1"],
                "non_empty_f1": char_stat["non_empty_f1"],
                "seq_f1": seq_stat["f1"],
                "transitions": model.crf.transitions.detach().cpu().clone(),
            }

    best["elapsed"] = time.perf_counter() - t0
    return best


part_c_rows = [["Tokenizer", "Scheme", "BestEp", "Char F1", "Empty F1", "NonEmpty F1", "Seqeval F1", "Time(s)"]]
crf_models_info = {}

for tok_name, tok in tokenizers.items():
    for scheme_name in SCHEMES:
        res = run_bilstm_crf(tok_name, tok, scheme_name)
        crf_models_info[(tok_name, scheme_name)] = res
        part_c_rows.append([
            tok_name,
            scheme_name,
            str(res["epoch"]),
            f"{res['char_f1']:.5f}",
            f"{res['empty_f1']:.5f}",
            f"{res['non_empty_f1']:.5f}",
            f"{res['seq_f1']:.5f}",
            f"{res['elapsed']:.1f}",
        ])

log("CHECKPOINT 4", "Bagian C (BiLSTM-CRF) selesai")
print_table("Bagian C: Probe Sekuensial Ringan (BiLSTM-CRF di tsd_trial)", part_c_rows)

for scheme_name in SCHEMES:
    labels = list(SCHEMES[scheme_name].keys())
    mat = crf_models_info[("BERT-WordPiece", scheme_name)]["transitions"].numpy()
    rows = [["From \\ To"] + labels]
    for i, src_lbl in enumerate(labels):
        rows.append([src_lbl] + [f"{mat[i, j]:+.2f}" for j in range(len(labels))])
    print()
    print_table(f"Matriks Transisi CRF (BERT-WordPiece, {scheme_name})", rows)

[CHECKPOINT 4] Bagian C (BiLSTM-CRF) selesai
Bagian C: Probe Sekuensial Ringan (BiLSTM-CRF di tsd_trial)
------------------------------------------------------------------------------------------
Tokenizer      | Scheme | BestEp | Char F1 | Empty F1 | NonEmpty F1 | Seqeval F1 | Time(s)
BERT-WordPiece | BIO    | 6      | 0.61415 | 0.16279  | 0.64415     | 0.59268    | 507.0
BERT-WordPiece | BIOES  | 6      | 0.62127 | 0.13953  | 0.65329     | 0.59042    | 499.9
RoBERTa-BPE    | BIO    | 4      | 0.59045 | 0.13953  | 0.62042     | 0.57143    | 507.5
RoBERTa-BPE    | BIOES  | 6      | 0.60246 | 0.18605  | 0.63014     | 0.57213    | 508.9
XLM-R-SPM      | BIO    | 6      | 0.62269 | 0.09302  | 0.65789     | 0.56810    | 536.7
XLM-R-SPM      | BIOES  | 4      | 0.61187 | 0.11628  | 0.64481     | 0.58845    | 574.4

Matriks Transisi CRF (BERT-WordPiece, BIO)
-------------------------------------------------
From \ To | O     | B-TOXIC | I-TOXIC
O         | +1.16 | +0.75   | -1.53
B-TOXIC   |

## Panduan Membaca Hasil Eksperimen

1. **Bagian A (*Round-Trip Ceiling*):** Pastikan seluruh kombinasi lulus ambang $\ge 0.96$. Perhatikan apakah `RoBERTa-BPE` atau `XLM-R-SPM` memiliki subword/span lebih banyak daripada `BERT-WordPiece` akibat pemisahan kosakata multibahasa/cased.
2. **Bagian B (`SGDClassifier`):** Bandingkan antar-tokenizer pada `BIO` saja untuk melihat apakah potongan subword salah satu tokenizer lebih mudah diklasifikasikan secara leksikal.
3. **Bagian C (`BiLSTM-CRF`):** Fokus pada delta `BIO` vs `BIOES` (terutama pada `Char F1`, `NonEmpty F1`, dan `Seqeval F1`) serta nilai matriks transisi (misalnya apakah `S-TOXIC -> I-TOXIC` dan `O -> I-TOXIC` belajar bobot negatif). Kombinasi terbaik menjadi kandidat utama untuk diuji dengan pretrained Transformer penuh di Phase 4.